# Download and Inspect Existing EV Charging Infrastructure

## Goal

Download the current U.S. Department of Energy Alternative Fuels Data Center (AFDC) public, available EV charging-unit inventory for Georgia. Preserve the raw snapshot, inspect the fields needed for this project, and create a smaller analysis-ready table.

Official source: https://developer.nlr.gov/docs/transportation/alt-fuel-stations-v1/ev-charging-units/

> Each row represents a charging unit. A station may therefore appear in multiple rows. The seven-county study area will be selected later with a spatial join rather than an unreliable city-name filter.

## Setup

This notebook requests electric stations in Georgia that are currently available and publicly accessible. The charging-unit endpoint is used because it includes connector-specific port counts and power output in kilowatts.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json

import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\cason\GIS_Portfolio_3\Project 4 - EV Charging Suitability")
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "charging_stations"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

download_date = datetime.now(timezone.utc).date().isoformat()
raw_csv = RAW_DIR / f"afdc_ev_charging_units_ga_{download_date}.csv"
metadata_json = RAW_DIR / f"afdc_ev_charging_units_ga_{download_date}_metadata.json"
processed_csv = PROCESSED_DIR / "afdc_ev_charging_units_ga_core.csv"

parameters = {
    "api_key": "DEMO_KEY",
    "fuel_type": "ELEC",
    "state": "GA",
    "status": "E",
    "access": "public",
    "limit": "all",
}
endpoint = "https://developer.nlr.gov/api/alt-fuel-stations/v1/ev-charging-units.csv"
request_url = f"{endpoint}?{urlencode(parameters)}"

print("Raw snapshot:", raw_csv)
print("Processed table:", processed_csv)

Raw snapshot: C:\Users\cason\GIS_Portfolio_3\Project 4 - EV Charging Suitability\data\raw\charging_stations\afdc_ev_charging_units_ga_2026-07-26.csv
Processed table: C:\Users\cason\GIS_Portfolio_3\Project 4 - EV Charging Suitability\data\processed\afdc_ev_charging_units_ga_core.csv


## Steps

### 1. Download and preserve the raw AFDC snapshot

In [2]:
request = Request(request_url, headers={"User-Agent": "GIS-Portfolio-EV-Suitability/1.0"})
with urlopen(request, timeout=120) as response:
    raw_csv.write_bytes(response.read())

metadata = {
    "source": "U.S. DOE Alternative Fuels Data Center",
    "documentation": "https://developer.nlr.gov/docs/transportation/alt-fuel-stations-v1/ev-charging-units/",
    "endpoint": endpoint,
    "parameters_without_api_key": {key: value for key, value in parameters.items() if key != "api_key"},
    "downloaded_utc": datetime.now(timezone.utc).isoformat(),
    "raw_file": raw_csv.name,
}
metadata_json.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Downloaded {raw_csv.stat().st_size:,} bytes")
print("Metadata saved:", metadata_json.name)

Downloaded 2,881,023 bytes
Metadata saved: afdc_ev_charging_units_ga_2026-07-26_metadata.json


### 2. Inspect the fields and create the core table

In [3]:
stations = pd.read_csv(raw_csv, low_memory=False)

requested_fields = [
    "ID", "Station Name", "Street Address", "City", "State", "ZIP",
    "Status Code", "Access Code", "Restricted Access", "Facility Type",
    "Latitude", "Longitude", "EV Network", "EV Connector Types",
    "EV Level1 EVSE Num", "EV Level2 EVSE Num", "EV DC Fast Count",
    "EV J1772 Connector Count", "EV J1772 Power Output (kW)",
    "EV CCS Connector Count", "EV CCS Power Output (kW)",
    "EV CHAdeMO Connector Count", "EV CHAdeMO Power Output (kW)",
    "EV J3400 Connector Count", "EV J3400 Power Output (kW)",
    "Date Last Confirmed", "Updated At", "Open Date",
]

available_fields = [field for field in requested_fields if field in stations.columns]
missing_fields = [field for field in requested_fields if field not in stations.columns]
core = stations.loc[:, available_fields].copy()
core.to_csv(processed_csv, index=False)

print(f"Rows (charging units): {len(stations):,}")
print(f"Unique station IDs: {stations['ID'].nunique():,}")
print(f"Columns in raw file: {len(stations.columns):,}")
print(f"Core fields retained: {len(available_fields):,}")
print("Missing requested fields:", missing_fields or "None")
print("Saved:", processed_csv)

Rows (charging units): 7,531
Unique station IDs: 2,375
Columns in raw file: 86
Core fields retained: 28
Missing requested fields: None
Saved: C:\Users\cason\GIS_Portfolio_3\Project 4 - EV Charging Suitability\data\processed\afdc_ev_charging_units_ga_core.csv


## Checks

Confirm the download contains only Georgia electric stations with valid coordinates, then summarize charging levels, connectors, and power-field coverage.

In [4]:
assert not stations.empty, "AFDC returned no records."
assert stations["State"].dropna().eq("GA").all(), "The extract contains records outside Georgia."
assert stations["Latitude"].notna().all() and stations["Longitude"].notna().all(), "Coordinates are missing."

level_fields = [
    field for field in ["EV Level1 EVSE Num", "EV Level2 EVSE Num", "EV DC Fast Count"]
    if field in stations.columns
]
connector_count_fields = [field for field in stations.columns if field.endswith("Connector Count")]
power_fields = [field for field in stations.columns if "Power Output (kW)" in field]

summary = pd.DataFrame({
    "non_null_records": stations[level_fields + connector_count_fields + power_fields].notna().sum(),
    "reported_total": stations[level_fields + connector_count_fields].fillna(0).sum()
        .reindex(level_fields + connector_count_fields + power_fields),
})

print("All validation checks passed.")
display(summary.fillna("—"))

All validation checks passed.


,non_null_records,reported_total
EV Level1 EVSE Num,88,3874.0
EV Level2 EVSE Num,5334,33021.0
EV DC Fast Count,2297,23170.0
EV J1772 Connector Count,7531,4763.0
EV CCS Connector Count,7531,1107.0
EV CHAdeMO Connector Count,7531,270.0
EV J3400 Connector Count,7531,1855.0
EV J3271 Connector Count,7531,0.0
EV J1772 Power Output (kW),4344,—
EV CCS Power Output (kW),1107,—


## Next Steps

1. Download Census county boundaries for Georgia.
2. Convert station coordinates to a spatial layer.
3. Spatially select Gwinnett, Barrow, Hall, Jackson, Clarke, Oconee, and Walton counties.
4. Produce the first infrastructure map and county-level port inventory.